In [1]:
# pip install pytorch_tabnet

In [ ]:
import sys
import sklearn
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from pytorch_tabnet.tab_model import TabNetRegressor
import os
import pandas as pd
from sklearn.metrics import mean_absolute_error

In [3]:
car_df = pd.read_csv('data_ingestion_zack/merged_output.csv')

In [4]:
car_df_2 = car_df[car_df['source'] != 'UCI']
car = car_df_2.drop(columns=['source'])
car = car.dropna()
car_small = car.sample(n=1000, random_state=42)

In [5]:
car_small

,make,model,year,engine_size,mileage,fuel_type,transmission,price
269223,Volkswagen,Caddy Maxi Life,2012.0,1.6,60178.0,Diesel,Manual,9995.000000
8838,Honda,Model D,2000.0,2.1,173468.0,Petrol,Manual,14639.483156
265070,Volkswagen,Touran,2003.0,1.9,162120.0,Diesel,Manual,990.000000
179359,Audi,Q7,2015.0,3.0,34752.0,Diesel,Automatic,29289.000000
160661,Renault,Twingo,2009.0,1.6,99773.0,Petrol,Manual,2790.000000
...,...,...,...,...,...,...,...,...
17826,Citroen,C2,2008.0,1.1,79000.0,Petrol,Manual,1999.000000
71571,Jaguar,E-PACE,2018.0,2.0,4400.0,Petrol,Automatic,35990.000000
239428,Vauxhall,Zafira,2008.0,1.6,123000.0,Petrol,Manual,995.000000
213270,Bmw,X5,2012.0,3.0,90716.0,Diesel,Automatic,16290.000000


In [7]:
print(car.shape)

(273456, 8)


In [8]:
X = car_small.drop(columns=['price', 'make', 'model', 'fuel_type', 'transmission'])
y = car_small['price']
X = pd.get_dummies(X)
num_features = X.shape[1]

X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=42)
X_valid, X_test, y_valid, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

model = TabNetRegressor(
    n_d=32,
    n_a=32,
    n_steps=5,
    gamma=1.5
)

/opt/anaconda3/lib/python3.13/site-packages/pytorch_tabnet/abstract_model.py:82: UserWarning: Device used : cpu
  warnings.warn(f"Device used : {self.device}")


In [9]:
X_train = X_train.values
X_valid = X_valid.values
X_test = X_test.values

y_train = y_train.values.reshape(-1,1)
y_valid = y_valid.values.reshape(-1,1)
y_test = y_test.values.reshape(-1,1)

In [10]:
model.fit(
    X_train, y_train,
    eval_set=[(X_valid, y_valid)],
    eval_metric=['mae'],
    max_epochs=20,
    patience=10,
    batch_size=128
)

epoch 0  | loss: 530245100.8| val_0_mae: 26905.43864|  0:00:00s
epoch 1  | loss: 573427392.0| val_0_mae: 100942.69294|  0:00:00s
epoch 2  | loss: 551947948.8| val_0_mae: 57121.83893|  0:00:00s
epoch 3  | loss: 553752249.6| val_0_mae: 81255.76706|  0:00:00s
epoch 4  | loss: 569776940.8| val_0_mae: 65337.49945|  0:00:00s
epoch 5  | loss: 565326585.6| val_0_mae: 72160.68671|  0:00:00s
epoch 6  | loss: 574368620.8| val_0_mae: 44616.29691|  0:00:00s
epoch 7  | loss: 547250233.6| val_0_mae: 90011.48717|  0:00:00s
epoch 8  | loss: 563111270.4| val_0_mae: 75324.30952|  0:00:00s
epoch 9  | loss: 540101241.6| val_0_mae: 81811.66704|  0:00:00s
epoch 10 | loss: 553016294.4| val_0_mae: 94236.22851|  0:00:00s

Early stopping occurred at epoch 10 with best_epoch = 0 and best_val_0_mae = 26905.43864


/opt/anaconda3/lib/python3.13/site-packages/pytorch_tabnet/callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)


In [11]:
preds = model.predict(X_test)

mae = mean_absolute_error(y_test, preds)
print(mae)

24981.504802929292
